# Subset Sum Oracle Based on arXiv:2410.01775 and `quantum-place-route`

**Status: unfinished working draft.**

This notebook currently follows the paper and its public repository code as a reference implementation. The next goal is to turn it into a more blog-style walkthrough, with more explanation of how the subset-sum summation and oracle construction work at the circuit level.

This notebook follows the public implementation released with:

**Angelo Benoit, Sam Schwartz, Ron K. Cytron,  
"Optimization of a Quantum Subset Sum Oracle", arXiv:2410.01775.**

Reference code: `ABenoit0226/quantum-place-route`, especially `adder_logic_script.ipynb`.

The small instance here is:

```python
S = [1, 2, 3]
T = 3
```

The matching subsets are `[3]` and `[1, 2]`.


## Code Map

The public notebook organizes the oracle construction around three pieces:

- `QC`: a light wrapper around `QuantumCircuit` that allocates one-qubit registers and basic reversible gates.
- `QVarArith`: little-endian quantum integers, addition, bit comparison, and all-zero checking.
- `QSubsetSum`: subset selector qubits, shadow registers, running sums, and target comparison.

This notebook keeps that structure and modernizes the simulator calls for current Qiskit/Aer.


In [ ]:
# If needed:
# !pip install qiskit qiskit-aer matplotlib pylatexenc

from __future__ import annotations

from itertools import product
from math import log2

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit_aer import AerSimulator


## Problem Instance

The selector register `x` has one qubit per set element.  
`x[i] = 1` means the value `S[i]` is included in the subset.


In [ ]:
S = [1, 2, 3]
T = 3

solutions = []
for bits in product([0, 1], repeat=len(S)):
    total = sum(bit * value for bit, value in zip(bits, S))
    if total == T:
        solutions.append(bits)

print("S =", S)
print("T =", T)
print("Classical solutions in x[0], x[1], x[2] order:", solutions)


## Little-Endian Registers

The repo code stores integers least-significant bit first.  
For example, decimal `3` with width `3` is binary `011`, stored as `[1, 1, 0]`.


In [ ]:
def num_bits(value: int) -> int:
    """Number of bits needed for a nonnegative integer."""
    return max(1, int(log2(value)) + 1) if value > 0 else 1


def little_endian_bits(value: int, width: int) -> list[int]:
    """Return [bit_0, bit_1, ..., bit_{width-1}]."""
    return [(value >> j) & 1 for j in range(width)]


print("3 with width 3 ->", little_endian_bits(3, 3))
print("sum(S) bit width ->", num_bits(sum(S)))


## Circuit Wrapper

This follows the `QC` helper in the public repo.  
It allocates named one-qubit registers and exposes the small reversible logic blocks used by the arithmetic layer.


In [ ]:
class QC:
    """Small allocator/wrapper around Qiskit's QuantumCircuit."""

    def __init__(self, gen_barriers: bool = False):
        self.reg_num = 0
        self.gen_barriers = gen_barriers
        self.circuit = QuantumCircuit()
        self.false_bit = self.add_bit("false")
        self.true_bit = self.add_bit("true")
        self.circuit.x(self.true_bit)

    def next_name(self, prefix: str) -> str:
        name = f"{prefix}_{self.reg_num}"
        self.reg_num += 1
        return name

    def add_bit(self, prefix: str = "q"):
        reg = QuantumRegister(1, self.next_name(prefix))
        self.circuit.add_register(reg)
        return reg[0]

    def add_classical_bits(self, width: int, name: str):
        reg = ClassicalRegister(width, name)
        self.circuit.add_register(reg)
        return reg

    def measure_bits(self, qubits, name: str):
        creg = self.add_classical_bits(len(qubits), name)
        for i, qubit in enumerate(qubits):
            self.circuit.measure(qubit, creg[i])
        return creg

    def barrier(self):
        if self.gen_barriers:
            self.circuit.barrier()

    def output_bit(self, output, prefix: str):
        return output if output is not None else self.add_bit(prefix)

    def qxor(self, bit1, bit2, output=None):
        output = self.output_bit(output, "xor")
        self.circuit.cx(bit1, output)
        self.circuit.cx(bit2, output)
        return output

    def qxor3(self, bit1, bit2, bit3, output=None):
        output = self.output_bit(output, "xor3")
        self.circuit.cx(bit1, output)
        self.circuit.cx(bit2, output)
        self.circuit.cx(bit3, output)
        return output

    def quarry(self, bit1, bit2, bit3, output=None):
        """Carry bit for adding three one-bit values."""
        output = self.output_bit(output, "carry")
        self.circuit.ccx(bit1, bit2, output)
        self.circuit.ccx(bit2, bit3, output)
        self.circuit.ccx(bit1, bit3, output)
        return output

    def same_xor(self, bit1, bit2, output=None):
        """XOR result: 0 means the two bits are equal."""
        output = self.output_bit(output, "same")
        self.circuit.cx(bit1, output)
        self.circuit.cx(bit2, output)
        return output


## Variable-Width Arithmetic

The paper and repo reduce qubits by using only as many bits as each integer or partial sum needs.  
`QVarArith` implements little-endian constants, temporary registers, addition, and equality comparison.


In [ ]:
class QVarArith:
    """Variable-width little-endian arithmetic helpers."""

    def __init__(self, qc: QC):
        self.qc = qc
        self.one = self.qint(1, name="one")

    def qint(self, value: int, name: str = "int", width: int | None = None):
        width = num_bits(value) if width is None else width
        bits = []
        for bit in little_endian_bits(value, width):
            qubit = self.qc.add_bit(name)
            if bit:
                self.qc.circuit.x(qubit)
            bits.append(qubit)
        return bits

    def qtemp(self, value: int, name: str = "tmp"):
        return [self.qc.add_bit(name) for _ in range(num_bits(value))]

    def onebit(self, carry_in, bit_a, bit_b):
        sum_bit = self.qc.qxor3(carry_in, bit_a, bit_b)
        carry_out = self.qc.quarry(carry_in, bit_a, bit_b)
        return sum_bit, carry_out

    def onebit_short(self, carry_in, bit_a):
        sum_bit = self.qc.qxor(carry_in, bit_a)
        carry_out = self.qc.add_bit("carry")
        self.qc.circuit.ccx(carry_in, bit_a, carry_out)
        return sum_bit, carry_out

    def add(self, val1, val2, width: int):
        """Return a new register holding val1 + val2 up to the requested width."""
        if len(val1) > len(val2):
            return self.add(val2, val1, width)

        result = []
        carry = self.qc.false_bit

        for i in range(len(val2)):
            if i < len(val1):
                sum_bit, carry = self.onebit(carry, val1[i], val2[i])
            else:
                sum_bit, carry = self.onebit_short(carry, val2[i])
            result.append(sum_bit)

        while len(result) < width:
            result.append(carry)
            carry = self.qc.add_bit("pad")

        return result[:width]

    def compare_bits(self, val1, val2):
        """Bitwise XOR comparison. A 0 bit means equality at that position."""
        width = max(len(val1), len(val2))
        padded_1 = list(val1) + [self.qc.false_bit] * (width - len(val1))
        padded_2 = list(val2) + [self.qc.false_bit] * (width - len(val2))
        return [self.qc.same_xor(padded_1[i], padded_2[i]) for i in range(width)]

    def all_zeros(self, bits, output=None):
        """Toggle output when every bit in bits is 0."""
        output = self.qc.output_bit(output, "all_zero")
        for bit in bits:
            self.qc.circuit.x(bit)

        if len(bits) == 1:
            self.qc.circuit.cx(bits[0], output)
        else:
            self.qc.circuit.mcx(bits, output)

        for bit in bits:
            self.qc.circuit.x(bit)
        return output

    def qcompare(self, val1, val2):
        return self.all_zeros(self.compare_bits(val1, val2))


## Subset Sum Oracle Builder

This mirrors the repo's `QSubsetSum.run` method:

1. allocate selector qubits `x`,
2. put `x` into uniform superposition,
3. allocate each integer `a_i`,
4. build a shadow register `b_i = a_i AND x_i`,
5. add shadows into a running sum,
6. compare the final sum with target `T`.

The flags `sorted_values`, `partial`, `variable`, and `width` correspond to the repo's experiment knobs.


In [ ]:
class QSubsetSum:
    """Subset Sum oracle construction based on the public repo structure."""

    def __init__(self, values: list[int], target: int, qc: QC | None = None):
        self.qc = QC() if qc is None else qc
        self.values = list(values)
        self.target = target
        self.x = []
        self.answer = None
        self.result = None
        self.qi = None

    def run(
        self,
        sorted_values: bool = True,
        partial: bool = True,
        variable: bool = True,
        width: int | None = None,
        measure: bool = True,
    ):
        values = sorted(self.values) if sorted_values else list(self.values)
        width_of_sum = num_bits(sum(values)) if width is None else width
        qint_width = None if variable else width_of_sum
        self.qi = QVarArith(self.qc)

        self.x = []
        for _ in values:
            selector = self.qc.add_bit("x")
            self.x.append(selector)
            self.qc.circuit.h(selector)

        running_sum = self.qi.qint(0, name="sum")
        total_so_far = 0 if partial else sum(values)

        for i, value in enumerate(values):
            value_reg = self.qi.qint(value, name="a", width=qint_width)
            shadow = self.qi.qtemp(value, name="shd")

            for j in range(len(shadow)):
                self.qc.circuit.ccx(self.x[i], value_reg[j], shadow[j])

            if partial:
                total_so_far += value

            sum_width = num_bits(total_so_far) if width is None else qint_width
            running_sum = self.qi.add(running_sum, shadow, sum_width)

        target_reg = self.qi.qint(self.target, name="target", width=len(running_sum))
        result = self.qi.qcompare(target_reg, running_sum)

        self.answer = running_sum
        self.result = result

        if measure:
            self.qc.measure_bits(self.x, "x")
            self.qc.measure_bits(running_sum, "sum")
            self.qc.measure_bits([result], "ok")

        return running_sum, result

    def circuit(self):
        return self.qc.circuit

    def counts(self, shots: int = 1024):
        simulator = AerSimulator(method="matrix_product_state")
        compiled = transpile(self.circuit(), basis_gates=["x", "h", "cx", "ccx", "u", "p", "measure"], optimization_level=0)
        return simulator.run(compiled, shots=shots).result().get_counts()

    def resources(self):
        circuit = self.circuit()
        return {
            "qubits": circuit.num_qubits,
            "classical_bits": circuit.num_clbits,
            "depth": circuit.depth(),
            "gate_counts": dict(circuit.count_ops()),
        }


## Run the Small Example

The `ok` classical bit is `1` when the computed sum equals the target.
The count keys are printed in Qiskit's register order, so this cell also prints the circuit resources directly.


In [ ]:
qsub = QSubsetSum(S, T)
qsub.run(sorted_values=True, partial=True, variable=True, measure=True)

print("Resources:")
for key, value in qsub.resources().items():
    print(f"  {key}: {value}")

counts = qsub.counts(shots=1024)
print()
print("Raw counts:")
for key, value in sorted(counts.items(), key=lambda item: item[1], reverse=True):
    print(f"  {key}: {value}")


## Compare the Repo Knobs

The paper and repo compare several construction choices:

- sorted vs unsorted set values,
- partial-sum widths vs fixed total-sum width,
- variable-width integers vs fixed-width integers.

For this tiny instance the differences are small, but the resource counters show why these choices matter for larger inputs.


In [ ]:
def build_and_report(values, target, *, sorted_values=True, partial=True, variable=True, width=None):
    qsub = QSubsetSum(values, target)
    qsub.run(
        sorted_values=sorted_values,
        partial=partial,
        variable=variable,
        width=width,
        measure=False,
    )
    resources = qsub.resources()
    return {
        "sorted": sorted_values,
        "partial_widths": partial,
        "variable_widths": variable,
        "qubits": resources["qubits"],
        "depth": resources["depth"],
        "gate_counts": resources["gate_counts"],
    }

variants = [
    build_and_report(S, T, sorted_values=True, partial=True, variable=True),
    build_and_report(S, T, sorted_values=False, partial=True, variable=True),
    build_and_report(S, T, sorted_values=False, partial=False, variable=True),
    build_and_report(S, T, sorted_values=True, partial=False, variable=False, width=num_bits(sum(S))),
]

for variant in variants:
    print(variant)


## Link Back to Grover

The circuit above builds the subset-sum predicate: selector values whose running sum equals `T` set the `ok` flag.  
A Grover implementation uses this predicate as the oracle core, adds phase kickback, uncomputes temporary registers, and applies the diffuser on selector qubits.
